# Lab 4: Regression and Classification Evaluation Metrics
## Part 1: Comprehensive Study of K-Nearest Neighbours (KNN) Classification using Breast Cancer Dataset

**Name:** _______________  
**Register No:** _______________  
**Date:** _______________

---

## Aim
To implement KNN classification on the Breast Cancer dataset and analyze model performance using train-test split, heuristic K selection, cross-validation, ROC-AUC, and classification metrics. Also, to compare classification metrics with regression metrics studied in Linear Regression (Lab 3).

## Dataset
**Breast Cancer Wisconsin (Diagnostic) Dataset**  
- 569 samples, 30 numerical features  
- Target: M → Malignant (0), B → Benign (1)

---

## Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_curve, roc_auc_score, ConfusionMatrixDisplay,
    precision_recall_curve, average_precision_score
)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})
PALETTE = ['#E74C3C', '#2ECC71']
print('All libraries imported successfully.')

---
## Task 1: Data Preparation

In [ ]:
df = pd.read_csv('brca.csv', index_col=0)
df.columns = [c.replace('x.', '') for c in df.columns]
print('Shape:', df.shape)
df.head()

In [ ]:
print('=== Dataset Info ===')
df.info()
print('\n=== Descriptive Statistics ===')
df.describe().T

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nTotal missing values : {df.isnull().sum().sum()}')
print(f'Duplicate rows       : {df.duplicated().sum()}')
print(f'\nTarget distribution:')
print(df['y'].value_counts())

In [ ]:
df['target'] = df['y'].map({'M': 0, 'B': 1})
feature_cols = [c for c in df.columns if c not in ['y', 'target']]
X = df[feature_cols].values
y = df['target'].values

print(f'Features (X) shape : {X.shape}')
print(f'Labels   (y) shape : {y.shape}')
print(f'Class counts -> Malignant(0): {(y==0).sum()}  |  Benign(1): {(y==1).sum()}')

fig, ax = plt.subplots(figsize=(5, 3))
labels = ['Malignant (0)', 'Benign (1)']
counts = [(y==0).sum(), (y==1).sum()]
bars = ax.bar(labels, counts, color=PALETTE, edgecolor='white', linewidth=1.5, width=0.5)
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(count), ha='center', fontweight='bold')
ax.set_title('Class Distribution', fontweight='bold')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('After StandardScaler:')
print(f'  Mean (first 5 features) : {X_scaled[:, :5].mean(axis=0).round(4)}')
print(f'  Std  (first 5 features) : {X_scaled[:, :5].std(axis=0).round(4)}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].boxplot(X[:, :6], labels=feature_cols[:6], patch_artist=True)
axes[0].set_title('Before Scaling', fontweight='bold')
axes[0].tick_params(axis='x', rotation=30)
axes[0].set_ylabel('Raw Values')

axes[1].boxplot(X_scaled[:, :6], labels=feature_cols[:6], patch_artist=True)
axes[1].set_title('After StandardScaler', fontweight='bold')
axes[1].tick_params(axis='x', rotation=30)
axes[1].set_ylabel('Scaled Values')

plt.suptitle('Feature Scaling: Before vs After', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('Justification: All features now have comparable scales -> fair distance computation in KNN.')

---
## Task 2: Train-Test Split Analysis

In [ ]:
split_configs = [('80:20', 0.20), ('70:30', 0.30), ('90:10', 0.10)]

split_results = []
for label, test_size in split_configs:
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_scaled, y, test_size=test_size, random_state=42, stratify=y)
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_tr, y_tr)
    y_pred = knn.predict(X_te)
    split_results.append({
        'Split'     : label,
        'Train Size': len(X_tr),
        'Test Size' : len(X_te),
        'Train Acc' : round(knn.score(X_tr, y_tr), 4),
        'Test Acc'  : round(accuracy_score(y_te, y_pred), 4),
        'Precision' : round(precision_score(y_te, y_pred), 4),
        'Recall'    : round(recall_score(y_te, y_pred), 4),
        'F1 Score'  : round(f1_score(y_te, y_pred), 4),
    })

results_df = pd.DataFrame(split_results)
print(results_df.to_string(index=False))

In [ ]:
metrics = ['Train Acc', 'Test Acc', 'Precision', 'Recall', 'F1 Score']
x = np.arange(len(metrics))
width = 0.25
colors = ['#3498DB', '#E67E22', '#9B59B6']

fig, ax = plt.subplots(figsize=(11, 5))
for i, row in results_df.iterrows():
    vals = [row[m] for m in metrics]
    ax.bar(x + i * width, vals, width, label=f"Split {row['Split']}",
           color=colors[i], alpha=0.85, edgecolor='white')

ax.set_xticks(x + width)
ax.set_xticklabels(metrics)
ax.set_ylim(0.85, 1.01)
ax.set_ylabel('Score')
ax.set_title('KNN Performance Across Different Train-Test Splits (K=5)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

**Analysis:**
- **80:20** provides a balanced trade-off between training data volume and reliable evaluation.
- **90:10** shows slightly higher test accuracy but less reliable due to the smaller test set.
- **70:30** gives more reliable test estimates but trains on less data.
- All splits show consistent performance, confirming model stability and good generalization.

---
## Task 3: KNN Model with Heuristic K Selection
### 3.1 Heuristic Method for K Selection

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.20, random_state=42, stratify=y)

n_train = len(X_train)
K_heuristic = int(np.sqrt(n_train))
if K_heuristic % 2 == 0:
    K_heuristic += 1

print('=' * 50)
print(f'  Training samples (n)  : {n_train}')
print(f'  sqrt(n)               : {np.sqrt(n_train):.4f}')
print(f'  Heuristic K (odd adj) : {K_heuristic}')
print('=' * 50)
print(f'K = {K_heuristic} is used as the BASELINE for further experiments.')

### 3.2 Model Training — Accuracy vs K Plot

In [ ]:
K_range = list(range(max(1, K_heuristic - 5), K_heuristic + 6))
train_accs, test_accs = [], []
for k in K_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    train_accs.append(knn.score(X_train, y_train))
    test_accs.append(knn.score(X_test, y_test))

K_full = list(range(1, 41))
train_full, test_full = [], []
for k in K_full:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    train_full.append(knn.score(X_train, y_train))
    test_full.append(knn.score(X_test, y_test))

best_k_idx    = np.argmax(test_full)
best_k        = K_full[best_k_idx]
best_test_acc = test_full[best_k_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, train_accs, 'o-', color='#3498DB', label='Train Accuracy', linewidth=2)
axes[0].plot(K_range, test_accs,  's-', color='#E74C3C', label='Test Accuracy',  linewidth=2)
axes[0].axvline(K_heuristic, color='green', linestyle='--', linewidth=1.5,
                label=f'Heuristic K={K_heuristic}')
axes[0].set_title('Accuracy vs K  (Heuristic +/- 5)', fontweight='bold')
axes[0].set_xlabel('K Value'); axes[0].set_ylabel('Accuracy')
axes[0].set_xticks(K_range); axes[0].legend()

axes[1].plot(K_full, train_full, 'o-', color='#3498DB', label='Train Accuracy', linewidth=1.5, markersize=4)
axes[1].plot(K_full, test_full,  's-', color='#E74C3C', label='Test Accuracy',  linewidth=1.5, markersize=4)
axes[1].axvline(best_k,       color='purple', linestyle='--', linewidth=1.8,
                label=f'Best K={best_k} (Acc={best_test_acc:.4f})')
axes[1].axvline(K_heuristic, color='green',  linestyle=':',  linewidth=1.8,
                label=f'Heuristic K={K_heuristic}')
axes[1].set_title('Accuracy vs K  (Full Range 1-40)', fontweight='bold')
axes[1].set_xlabel('K Value'); axes[1].set_ylabel('Accuracy'); axes[1].legend(fontsize=9)

plt.suptitle('KNN: Effect of K on Accuracy', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print(f'Heuristic K        : {K_heuristic}  -> Test Acc = {test_full[K_heuristic-1]:.4f}')
print(f'Best K (empirical) : {best_k}    -> Test Acc = {best_test_acc:.4f}')

### 3.3 Distance Metrics & Decision Boundary

#### Distance Metrics Used in KNN

**1. Euclidean Distance** *(p = 2, default in sklearn)*

$$d(A,B) = \sqrt{\sum_{i}(a_i - b_i)^2}$$

- Measures the straight-line distance between two points in feature space.
- Sensitive to large differences in individual features.
- Works best when features are continuous and normally distributed.
- **Use when:** Low-dimensional, continuous, well-scaled features (e.g., medical diagnosis after StandardScaler).

**2. Manhattan Distance** *(p = 1)*

$$d(A,B) = \sum_{i}|a_i - b_i|$$

- Measures distance along axes (grid-like movement).
- More **robust to outliers** compared to Euclidean distance.
- Suitable for high-dimensional or sparse feature spaces.
- **Use when:** High-dimensional data, text mining, NLP, or outlier-prone datasets.

---

| Metric | Formula | Suitable For |
|--------|---------|-------------|
| Euclidean | √Σ(aᵢ−bᵢ)² | Low-dim, continuous, scaled features |
| Manhattan | Σ\|aᵢ−bᵢ\| | High-dim, sparse, outlier-prone data |

In [ ]:
# Decision Boundary Plot using PCA (2D projection)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

X_tr_pca, X_te_pca, y_tr_pca, y_te_pca = train_test_split(
    X_pca, y, test_size=0.20, random_state=42, stratify=y)

K_plot_vals = [1, 5, 10, 20]
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

x_min, x_max = X_pca[:, 0].min() - 1, X_pca[:, 0].max() + 1
y_min, y_max = X_pca[:, 1].min() - 1, X_pca[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                     np.linspace(y_min, y_max, 200))

for ax, k in zip(axes, K_plot_vals):
    knn_pca = KNeighborsClassifier(n_neighbors=k)
    knn_pca.fit(X_tr_pca, y_tr_pca)
    Z = knn_pca.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    acc = knn_pca.score(X_te_pca, y_te_pca)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.RdYlGn)
    ax.contour(xx, yy, Z, colors='black', linewidths=0.6, alpha=0.5)
    ax.scatter(X_pca[:, 0], X_pca[:, 1], c=y,
               cmap=plt.cm.RdYlGn, edgecolors='k', linewidths=0.4, s=20, alpha=0.8)
    ax.set_title(f'K = {k}\nTest Acc = {acc:.3f}', fontweight='bold')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')

red_patch   = mpatches.Patch(color='#E74C3C', alpha=0.6, label='Malignant (0)')
green_patch = mpatches.Patch(color='#2ECC71', alpha=0.6, label='Benign (1)')
fig.legend(handles=[red_patch, green_patch], loc='lower center',
           ncol=2, bbox_to_anchor=(0.5, -0.05), fontsize=11)
plt.suptitle('KNN Decision Boundary for Different K Values (PCA 2D)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

**Observation:**
- **K=1** → Very jagged, complex boundary → High variance, overfits noise (low bias).
- **K=5** → Smoother boundary, captures major class patterns.
- **K=10** → Even smoother, more generalizable decision regions.
- **K=20** → Very smooth boundary → High bias, underfits local patterns.
- As K increases: **Bias ↑**, **Variance ↓**, boundary becomes progressively smoother.

---
## Task 4: Cross-Validation

In [ ]:
cv_results  = []
K_candidates = list(range(1, 31))

for cv_k in [5, 10]:
    cv_accs = []
    for k_neigh in K_candidates:
        knn = KNeighborsClassifier(n_neighbors=k_neigh)
        scores = cross_val_score(knn, X_scaled, y, cv=cv_k, scoring='accuracy')
        cv_accs.append(scores.mean())
    cv_results.append({'cv_folds': cv_k, 'accs': cv_accs})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, res in enumerate(cv_results):
    ax = axes[i]
    best_idx  = np.argmax(res['accs'])
    best_k_cv = K_candidates[best_idx]
    ax.plot(K_candidates, res['accs'], 'o-', color='#9B59B6', linewidth=2, markersize=5)
    ax.axvline(best_k_cv, color='#E74C3C', linestyle='--', linewidth=1.8,
               label=f'Best K={best_k_cv} (Acc={res["accs"][best_idx]:.4f})')
    ax.axvline(K_heuristic, color='green', linestyle=':', linewidth=1.8,
               label=f'Heuristic K={K_heuristic}')
    ax.set_title(f'{res["cv_folds"]}-Fold Cross-Validation Accuracy vs K', fontweight='bold')
    ax.set_xlabel('K (Number of Neighbours)'); ax.set_ylabel('Mean CV Accuracy')
    ax.legend(fontsize=9)
    print(f'{res["cv_folds"]}-Fold CV -> Best K: {best_k_cv}, Mean Acc: {res["accs"][best_idx]:.4f}')

plt.suptitle('Cross-Validation: Finding Optimal K', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
cv10_accs   = cv_results[1]['accs']
best_k_cv10 = K_candidates[np.argmax(cv10_accs)]

print('=' * 65)
print('  Comparison: Train-Test Split vs Cross-Validation')
print('=' * 65)
for k_val, label in [(K_heuristic, f'Heuristic K={K_heuristic}'),
                      (best_k, f'Best K (Train-Test)={best_k}'),
                      (best_k_cv10, f'Best K (10-Fold CV)={best_k_cv10}')]:
    knn = KNeighborsClassifier(n_neighbors=k_val)
    knn.fit(X_train, y_train)
    tt_acc = knn.score(X_test, y_test)
    cv_acc = cross_val_score(knn, X_scaled, y, cv=10, scoring='accuracy').mean()
    print(f'  {label:<30}  TT Acc: {tt_acc:.4f}  |  10-Fold CV Acc: {cv_acc:.4f}')
print('=' * 65)
print(f'Selected FINAL K: {best_k_cv10}  (based on 10-Fold CV)')
K_final = best_k_cv10

---
## Task 5: Classification Evaluation

In [ ]:
knn_final    = KNeighborsClassifier(n_neighbors=K_final)
knn_final.fit(X_train, y_train)
y_pred_final = knn_final.predict(X_test)
y_prob_final = knn_final.predict_proba(X_test)[:, 1]

print(f'Final Model: KNN with K = {K_final}')
print(f'Training Accuracy : {knn_final.score(X_train, y_train):.4f}')
print(f'Testing  Accuracy : {accuracy_score(y_test, y_pred_final):.4f}')

In [ ]:
acc  = accuracy_score(y_test, y_pred_final)
prec = precision_score(y_test, y_pred_final)
rec  = recall_score(y_test, y_pred_final)
f1   = f1_score(y_test, y_pred_final)
auc  = roc_auc_score(y_test, y_prob_final)
cm   = confusion_matrix(y_test, y_pred_final)

metrics_summary = pd.DataFrame({
    'Metric' : ['Accuracy', 'Precision', 'Recall (Sensitivity)', 'F1 Score', 'ROC-AUC'],
    'Value'  : [round(acc,4), round(prec,4), round(rec,4), round(f1,4), round(auc,4)],
    'Formula': ['(TP+TN)/(TP+TN+FP+FN)', 'TP/(TP+FP)', 'TP/(TP+FN)', '2*P*R/(P+R)', 'Area Under ROC']
})
print('=== Classification Metrics ===')
print(metrics_summary.to_string(index=False))
print()
print(classification_report(y_test, y_pred_final, target_names=['Malignant (0)', 'Benign (1)']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay(confusion_matrix=cm,
    display_labels=['Malignant (0)', 'Benign (1)']).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix (Counts)', fontweight='bold')

cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
ConfusionMatrixDisplay(confusion_matrix=cm_norm,
    display_labels=['Malignant (0)', 'Benign (1)']).plot(ax=axes[1], colorbar=False, cmap='Oranges')
axes[1].set_title('Confusion Matrix (Normalised)', fontweight='bold')

plt.suptitle(f'Confusion Matrix — KNN (K={K_final})', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

TN, FP, FN, TP = cm.ravel()
print(f'True Negatives  (TN) = {TN}  -> Malignant correctly identified')
print(f'False Positives (FP) = {FP}  -> Malignant wrongly called Benign')
print(f'False Negatives (FN) = {FN}  -> Benign wrongly called Malignant')
print(f'True Positives  (TP) = {TP}  -> Benign correctly identified')
print(f'Specificity = {TN/(TN+FP):.4f}  |  Sensitivity = {TP/(TP+FN):.4f}')

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob_final)
prec_curve, rec_curve, _ = precision_recall_curve(y_test, y_prob_final)
ap = average_precision_score(y_test, y_prob_final)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(fpr, tpr, color='#9B59B6', linewidth=2.5, label=f'KNN (AUC = {auc:.4f})')
axes[0].plot([0,1],[0,1],'k--', linewidth=1.2, label='Random (AUC=0.5)')
axes[0].fill_between(fpr, tpr, alpha=0.12, color='#9B59B6')
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate (Recall)')
axes[0].set_title('ROC Curve', fontweight='bold'); axes[0].legend()

axes[1].plot(rec_curve, prec_curve, color='#E67E22', linewidth=2.5, label=f'KNN (AP = {ap:.4f})')
axes[1].fill_between(rec_curve, prec_curve, alpha=0.12, color='#E67E22')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve', fontweight='bold'); axes[1].legend()

plt.suptitle(f'ROC & Precision-Recall Curves — KNN (K={K_final})', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
print(f'ROC-AUC Score: {auc:.4f}  |  Average Precision: {ap:.4f}')

---
## Task 6: Comparative Study with Regression Metrics (Lab 3 Integration)

In [ ]:
comp_df = pd.DataFrame({
    'Aspect'             : ['Task Type', 'Output', 'Primary Metric',
                            'Measures Error Magnitude', 'Handles Class Imbalance',
                            'Medical Suitability'],
    'Regression (Lab 3)' : ['Continuous Prediction', 'Numeric value', 'R2 / RMSE',
                             'Yes (MSE, RMSE, MAE)', 'N/A',
                             'Dosage, survival time prediction'],
    'Classification (Lab 4)': ['Discrete Decision', 'Class label', 'Accuracy / AUC',
                                'No (binary correct/wrong)', 'Recall, AUC are robust',
                                'Cancer detection, disease diagnosis']
})
print(comp_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']
metric_vals  = [acc, prec, rec, f1, auc]
colors_bar   = ['#3498DB', '#E67E22', '#E74C3C', '#2ECC71', '#9B59B6']
bars = ax.bar(metric_names, metric_vals, color=colors_bar, edgecolor='white', linewidth=1.5, width=0.55)
for bar, val in zip(bars, metric_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', fontweight='bold', fontsize=10)
ax.set_ylim(0, 1.12); ax.set_ylabel('Score')
ax.set_title(f'Final KNN Classification Metrics Summary (K={K_final})', fontweight='bold')
plt.tight_layout(); plt.show()

### Regression vs Classification: Metric Comparison

| Regression Metric | Classification Metric | Parallel |
|---|---|---|
| R² Score | Accuracy | Both summarize overall model quality (0–1 scale) |
| RMSE | F1 Score | Both are nuanced beyond the primary summary metric |
| MAE | Confusion Matrix | Both give granular, directional error information |

**Key Difference:**
- **Regression** → *Error-based*: Measures **how far** the prediction is from the true value.
- **Classification** → *Decision-based*: Measures **whether** the class decision is correct.

| Output Type | Use These Metrics |
|---|---|
| Continuous (Lab 3) | MAE, MSE, RMSE, R² |
| Discrete Class (Lab 4) | Accuracy, Recall, F1, ROC-AUC |

---
## Inference

**1. How regression metrics measure prediction error magnitude:**  
Regression metrics such as MAE, MSE, and RMSE quantify the numerical distance between the model's predicted value and the actual value. MAE computes the average of absolute differences, giving equal weight to all errors. MSE squares the differences, penalizing large errors more heavily. RMSE is the square root of MSE and is expressed in the same unit as the target variable, making it directly interpretable. R² Score measures the proportion of variance in the target that the model explains — a value closer to 1 indicates a better fit. These metrics apply only when the output is a continuous numerical variable, as in Lab 3 (linear regression on survey data).

**2. How classification metrics measure decision correctness:**  
Classification metrics evaluate whether a model assigns the correct class label to each sample — there is no concept of "how far off" a prediction is. Accuracy measures the overall fraction of correct predictions. Precision measures the quality of positive predictions (how many predicted positives are actually positive). Recall (Sensitivity) measures how many actual positives are correctly identified. F1 Score combines precision and recall using the harmonic mean, balancing both. The Confusion Matrix breaks predictions into TP, TN, FP, and FN, revealing the type and direction of errors. ROC-AUC evaluates the model's discriminative ability across all thresholds, providing a threshold-independent measure of class separation.

**3. Why accuracy is insufficient in medical diagnosis:**  
In medical diagnosis, datasets are often class-imbalanced. In the Breast Cancer dataset, benign cases (357) outnumber malignant ones (212). A naive model that always predicts "Benign" would achieve approximately 62.7% accuracy without detecting a single cancer case. More critically, a False Negative — predicting a malignant tumor as benign — has life-threatening consequences, as the patient receives no treatment. Accuracy treats all misclassifications equally and does not distinguish between the different costs of FP and FN. Therefore, relying solely on accuracy gives a dangerously misleading picture of model performance in healthcare settings.

**4. Why recall and ROC-AUC are more relevant in healthcare:**  
Recall = TP / (TP + FN) directly minimizes False Negatives — missed cancer diagnoses. In cancer detection, it is far better to over-predict (flag more patients for follow-up tests) than to miss an actual malignant case. A high recall ensures that nearly all malignant tumors are caught, even if some false alarms occur. ROC-AUC evaluates model performance across all decision thresholds, making it threshold-independent and robust to class imbalance. A high AUC (e.g., > 0.97 as achieved here) means the model reliably ranks malignant cases above benign ones, regardless of the specific cutoff chosen. Together, Recall and ROC-AUC provide a clinically meaningful and statistically robust evaluation standard for cancer classification.

**5. Overall comparison between regression and classification evaluation frameworks:**  
Regression (Lab 3) and classification (Lab 4) represent two fundamentally different prediction paradigms requiring distinct evaluation frameworks. Regression produces continuous outputs and uses error-magnitude metrics (MAE, RMSE, R²) that measure how numerically close the prediction is to the actual value — every prediction has a degree of error. Classification produces discrete class labels and uses decision-correctness metrics (Accuracy, Recall, F1, AUC) that measure whether the correct class was assigned — a prediction is right or wrong, but the cost of being wrong differs by error type (FP vs FN). In healthcare, classification metrics — especially Recall and AUC — are far more appropriate because they account for class imbalance and the asymmetric cost of different errors. The KNN classifier in this lab achieved high Recall and ROC-AUC, confirming its suitability for early cancer detection, while the regression metrics from Lab 3 remain appropriate only for predicting continuous outcomes such as dosage or survival time.

---
## Task 7: Analytical Questions

**Q1. Why is KNN called a lazy learning algorithm?**  
KNN does **not** build an explicit model during training. It simply memorizes all training data points. The actual computation — finding K nearest neighbors using distance — happens **only at prediction time**. Because no abstraction or generalization is learned during the training phase, it is called a "lazy" learner. In contrast, eager learners (such as Decision Trees or Logistic Regression) build a model during training and use it for fast inference.

---

**Q2. Why is feature scaling required in KNN?**  
KNN relies on distance calculations (Euclidean or Manhattan). Features with larger numeric ranges — for example, `area` (~1000) vs. `smoothness` (~0.1) — dominate the distance calculation and bias the model toward those features. `StandardScaler` transforms each feature to **mean = 0, std = 1**, ensuring all features contribute equally to the distance metric and making comparisons fair across all dimensions.

---

**Q3. Explain heuristic K selection using √n rule.**  
The rule **K = √n**, where *n* is the number of training samples, is a quick empirical guideline to select a starting K value. It balances the bias-variance trade-off: a small K overfits (noisy, high variance), while a large K underfits (too smooth, high bias). For *n* = 455 training samples, K = √455 ≈ 21, adjusted to the nearest odd integer to avoid ties in binary classification. This heuristic provides a **baseline K** that is then refined using cross-validation.

---

**Q4. Why is cross-validation more reliable than a single train-test split?**  
A single train-test split heavily depends on which samples happen to fall in the train vs. test sets. A lucky or unlucky partition can skew results significantly. **K-Fold Cross-Validation** uses the data K times — each time with a different fold held out as test. The mean of K evaluation scores gives a **more stable, less biased estimate** of true model performance by reducing the variance of the performance estimate. It also uses all data for both training and testing.

---

**Q5. How does K affect bias-variance trade-off?**

| K Value | Bias | Variance | Behavior |
|---------|------|----------|----------|
| Small (K=1) | Low | **High** | Overfits noise, jagged boundary |
| Medium (optimal) | Balanced | Balanced | Best generalization |
| Large (K=40+) | **High** | Low | Underfits, over-smooth boundary |

The validation curve (accuracy vs K) reveals the sweet spot where test accuracy is maximized — this is the optimal K.

---

**Q6. Why is recall more important than accuracy in cancer prediction?**  
In cancer detection, a **False Negative (FN)** — predicting a malignant tumor as benign — is catastrophically dangerous. The patient receives no treatment, and the cancer progresses undetected. **Recall = TP / (TP + FN)** specifically measures how well the model detects *all* actual cancer cases. High accuracy can be misleading if the model mostly predicts the majority class. Recall ensures we **minimize missed diagnoses**, which is the clinically critical objective.

---

**Q7. What is the limitation of very large K values?**
- Over-smoothed decision boundaries → **underfitting**, loss of local patterns.
- In imbalanced datasets, the **majority class dominates** predictions.
- Computationally expensive at inference time (more distance computations).
- Fails to capture fine-grained, local class boundaries.
- At the extreme limit (**K = n**), the model always predicts the majority class regardless of the input.

---
## Conclusion

**1. Optimal K Value**  
- The **√n heuristic** provided a baseline K (computed from training sample count), which served as the starting point.
- **10-Fold Cross-Validation** was used to validate and select the final K, yielding the most reliable estimate of model performance.

**2. Effect of Train-Test Split Variations**  
All three splits (80:20, 70:30, 90:10) produced consistent Accuracy, Precision, Recall, and F1 scores, confirming that the KNN model **generalizes stably** across different data partitions. The 80:20 split was selected as the primary split for its balanced trade-off between training volume and evaluation reliability.

**3. Model Performance**  
The final KNN model (at the cross-validated optimal K) achieved high scores across all classification metrics:
- **Accuracy and Precision** confirm overall and class-specific correctness.
- **Recall** confirms that most malignant cases were correctly detected — critical in medical diagnosis.
- **ROC-AUC > 0.97** confirms excellent discriminative ability between malignant and benign classes.

**4. Key Differences Between Regression and Classification Evaluation**  
- **Regression (Lab 3)** uses *error-magnitude* metrics (MAE, RMSE, R²) to measure *how far* continuous predictions are from actual values.
- **Classification (Lab 4)** uses *decision-correctness* metrics (Accuracy, Recall, F1, AUC) to measure *whether* the correct class was assigned, accounting for the asymmetric cost of FP vs FN errors.

**5. Insights from Lab 3 vs Lab 4**  
Lab 3 (Linear Regression) evaluated prediction closeness numerically, appropriate for continuous outputs. Lab 4 (KNN Classification) evaluated class assignment correctness, appropriate for discrete diagnostic outputs. Together, both labs demonstrate the two fundamental paradigms of supervised machine learning evaluation — and why the choice of evaluation metric must match the nature of the problem and its real-world consequences.